In [7]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, glob, shutil
sys.path.append("/content/drive/MyDrive/tcc")
from src import config

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

APARELHOS = {
    "samsung_a56":             ("celular", f"{config.BRUTOS}/celulares/samsung_a56"),
    "samsung_s23fe":           ("celular", f"{config.BRUTOS}/celulares/samsung_s23fe"),
    "canon_powershot_sx510hs": ("camera",  f"{config.BRUTOS}/cameras/canon_powershot_5x510hs"),
}
DATASET_V2 = f"{config.DRIVE}/dataset_v2.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
PADRAO = np.array([16, 11, 10, 16, 24, 40, 51, 61, 12, 12, 14, 19, 26, 58, 60, 55,
                   14, 13, 16, 24, 40, 57, 69, 56, 14, 17, 22, 29, 51, 87, 80, 62,
                   18, 22, 37, 56, 68, 109, 103, 77, 24, 35, 55, 64, 81, 104, 113, 92,
                   49, 64, 78, 87, 103, 121, 120, 101, 72, 92, 95, 98, 112, 100, 103, 99])

def qualidade(img):
    if img.format != "JPEG":
        return None
    escala = np.array(img.quantization[0]).mean() / PADRAO.mean() * 100
    return round((200 - escala) / 2 if escala <= 100 else 5000 / escala)

EDITORES = ("photoshop", "lightroom", "gimp", "snapseed", "picsart", "canva")

linhas = []
for aparelho, (tipo, pasta) in APARELHOS.items():
    if not os.path.isdir(pasta):
        print("Pasta não encontrada:", pasta)
        continue
    for f in sorted(os.listdir(pasta)):
        caminho = os.path.join(pasta, f)
        if not os.path.isfile(caminho):
            continue
        try:
            with Image.open(caminho) as img:
                exif = img.getexif()
                software = str(exif.get(305, ""))
                linhas.append({
                    "aparelho": aparelho, "tipo": tipo, "arquivo": f, "caminho": caminho,
                    "formato": img.format, "largura": img.width, "altura": img.height,
                    "qualidade": qualidade(img), "modelo_exif": str(exif.get(272, "")),
                    "editado": any(e in software.lower() for e in EDITORES),
                })
        except Exception as e:
            linhas.append({"aparelho": aparelho, "tipo": tipo, "arquivo": f,
                           "caminho": caminho, "formato": f"ERRO: {e}"})

inv = pd.DataFrame(linhas)
inv.to_csv(f"{config.BRUTOS}/inventario_v2_reais.csv", index=False)

print(inv.groupby("aparelho").agg(
    fotos=("arquivo", "size"),
    formatos=("formato", lambda x: ", ".join(sorted(set(map(str, x))))),
    menor_lado=("largura", lambda s: int(np.minimum(s, inv.loc[s.index, "altura"]).min())),
    qualidade_mediana=("qualidade", "median"),
    modelo_exif=("modelo_exif", lambda x: ", ".join(sorted(set(x)))),
).to_string())

problemas = inv[(inv["formato"] != "JPEG")
                | (inv[["largura", "altura"]].min(axis=1) < 1024)
                | (inv["editado"] == True)]
print(f"\nFotos com problema: {len(problemas)}")
if len(problemas):
    print(problemas[["aparelho", "arquivo", "formato", "largura", "altura", "editado"]].to_string())

                         fotos formatos  menor_lado  qualidade_mediana               modelo_exif
aparelho                                                                                        
canon_powershot_sx510hs     65     JPEG        2792               97.0  Canon PowerShot SX510 HS
samsung_a56                 99     JPEG        2252               96.0             Galaxy A56 5G
samsung_s23fe               44     JPEG        3000               96.0             Galaxy S23 FE

Fotos com problema: 0


In [6]:
print(inv[inv["modelo_exif"] == "SM-A536E"][["aparelho", "arquivo"]])

       aparelho              arquivo
64  samsung_a56  20241115_120145.jpg


In [9]:
inv_ok = inv[(inv["formato"] == "JPEG")
             & (inv[["largura", "altura"]].min(axis=1) >= 1024)
             & (inv["editado"] != True)].copy()

rng = np.random.default_rng(config.SEED)
inv_ok["particao_foto"] = "treino"
for aparelho, g in inv_ok.groupby("aparelho"):
    n_val = max(1, round(len(g) * 0.15))
    inv_ok.loc[rng.choice(g.index, n_val, replace=False), "particao_foto"] = "val"

print(pd.crosstab(inv_ok["aparelho"], inv_ok["particao_foto"]))

particao_foto            treino  val
aparelho                            
canon_powershot_sx510hs      55   10
samsung_a56                  84   15
samsung_s23fe                37    7


In [10]:
ZIP = f"{config.BRUTOS}/synthbuster/synthbuster.zip"
SB = "/content/synthbuster/synthbuster"
if not os.path.exists(SB):
    !unzip -q "{ZIP}" -d /content/synthbuster

pares = pd.read_csv(config.PARES_CSV)
treino_ids = pares[pares["split"] == "treino"]
escolhidos = treino_ids.sample(len(inv_ok), random_state=config.SEED)

DEST_EXTRA = f"{config.PADRONIZADOS}/original/ia_extra"
os.makedirs(DEST_EXTRA, exist_ok=True)
rng2 = np.random.default_rng(config.SEED + 1)

extras = []
for _, p in tqdm(escolhidos.iterrows(), total=len(escolhidos), desc="IA extra"):
    outros = [g for g in config.GERADORES_TREINO if g != p["gerador"]]
    gerador = outros[int(rng2.integers(len(outros)))]
    origem = glob.glob(f"{SB}/{gerador}/{p['id']}.*")[0]
    nome = f"{p['id']}_{gerador}.png"
    if not os.path.exists(f"{DEST_EXTRA}/{nome}"):
        shutil.copy(origem, f"{DEST_EXTRA}/{nome}")
    extras.append({"id": p["id"], "classe": "ia", "fonte": "synthbuster",
                   "aparelho_ou_gerador": gerador, "versao": "original",
                   "papel": "principal", "split": "treino", "grupo": "ia_extra",
                   "arquivo": f"dados_padronizados/original/ia_extra/{nome}"})

extras = pd.DataFrame(extras)
print(len(extras), "imagens de IA extras")
print(extras["aparelho_ou_gerador"].value_counts())

IA extra: 100%|██████████| 208/208 [00:08<00:00, 23.12it/s]

208 imagens de IA extras
aparelho_ou_gerador
stable-diffusion-1-3    36
stable-diffusion-xl     34
stable-diffusion-1-4    32
dalle3                  32
stable-diffusion-2      29
midjourney-v5           23
dalle2                  22
Name: count, dtype: int64


In [11]:
if os.path.exists(DATASET_V2):
    raise RuntimeError("O dataset_v2.csv já existe. Não rode esta célula de novo.")

df = pd.read_csv(config.DATASET_CSV)

base = df[(df["papel"] == "principal") & (df["versao"] == "original")].copy()
base["grupo"] = "base"

firefly = df[df["papel"] == "ood"].copy()
firefly["grupo"] = "firefly"

disp = pd.DataFrame({
    "id": [f"{a}_{i:03d}" for a, i in zip(inv_ok["aparelho"], inv_ok.groupby("aparelho").cumcount())],
    "classe": "real",
    "fonte": inv_ok["tipo"].values,
    "aparelho_ou_gerador": inv_ok["aparelho"].values,
    "versao": "nativo",
    "papel": "dispositivo",
    "split": "dispositivo",
    "grupo": "dispositivo",
    "particao_foto": inv_ok["particao_foto"].values,
    "arquivo": [os.path.relpath(c, config.DRIVE) for c in inv_ok["caminho"]],
})

v2 = pd.concat([base, extras, disp, firefly], ignore_index=True)
v2.to_csv(DATASET_V2, index=False)

print(v2.groupby(["grupo", "classe", "split"]).size().to_string())

grupo        classe  split      
base         ia      teste          150
                     treino         700
                     val            150
             real    teste          150
                     treino         700
                     val            150
dispositivo  real    dispositivo    208
firefly      ia      teste          150
                     treino         700
                     val            150
ia_extra     ia      treino         208
